## Step 1 – Reuse Week 4 artifacts and prepare a SQLite database

In Week 4, I already built a semantic search pipeline:

- PDF → text → chunks  
- Embeddings using `SentenceTransformer("all-MiniLM-L6-v2")`  
- FAISS vector index for fast similarity search  
- Saved artifacts:
  - `data/index/chunks.pkl`
  - `data/index/meta.pkl`
  - `data/index/faiss.index`

In Week 5, I will reuse this data and store the chunks in a **SQLite database**, which will later be combined with vector search to build a **hybrid (symbolic + semantic) retrieval system**.


In [31]:
# Step 1: Load Week 4 artifacts (chunks, metadata, FAISS index, embedding model)

from pathlib import Path
import pickle
import faiss
from sentence_transformers import SentenceTransformer

# Where the Week 4 artifacts live (adjust if needed)
INDEX_DIR = Path("data/index")
assert INDEX_DIR.exists(), f"INDEX_DIR not found: {INDEX_DIR.resolve()}"

# ---- Load chunks and metadata ----
with open(INDEX_DIR / "chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

with open(INDEX_DIR / "meta.pkl", "rb") as f:
    metadata = pickle.load(f)

print(f"Loaded {len(chunks)} chunks and {len(metadata)} metadata rows.")

# ---- Load FAISS index ----
faiss_index_path = INDEX_DIR / "faiss.index"
assert faiss_index_path.exists(), f"FAISS index not found at {faiss_index_path}"

faiss_index = faiss.read_index(str(faiss_index_path))
print("FAISS index dimension:", faiss_index.d)
print("FAISS index size (ntotal):", faiss_index.ntotal)

# ---- Load embedding model (same as Week 4) ----
model_name = "all-MiniLM-L6-v2"
embed_model = SentenceTransformer(model_name)
print(f"Loaded embedding model: {model_name}")



Loaded 198 chunks and 198 metadata rows.
FAISS index dimension: 384
FAISS index size (ntotal): 198
Loaded embedding model: all-MiniLM-L6-v2


In [32]:
# Step 2: Build / rebuild SQLite chunks table for hybrid retrieval

import sqlite3

from pathlib import Path

DB_PATH = Path("data/week5_hybrid.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Rebuilding database at:", DB_PATH.resolve())

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# ---- Create basic chunks table (id, pdf_name, text) ----
cur.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    id        INTEGER PRIMARY KEY,
    pdf_name  TEXT,
    text      TEXT
)
""")

# Optional: clear table if rebuilding
cur.execute("DELETE FROM chunks")

# ---- Insert all chunks ----
rows = []
for i, (chunk, meta) in enumerate(zip(chunks, metadata)):
    pdf_name = meta.get("pdf_name", "unknown.pdf")
    # chunk may be a str or dict; handle both cases
    if isinstance(chunk, dict):
        text = chunk.get("text", "")
    else:
        text = chunk
    rows.append((i, pdf_name, text))

cur.executemany(
    "INSERT INTO chunks (id, pdf_name, text) VALUES (?, ?, ?)",
    rows,
)
conn.commit()

print(f"Inserted {len(rows)} rows into 'chunks' table.")

# ---- Quick sanity check ----
cur.execute("SELECT id, pdf_name, substr(text, 1, 80) FROM chunks LIMIT 5")
for row in cur.fetchall():
    print(row)

cur.execute("SELECT COUNT(*) FROM chunks")
total_rows = cur.fetchone()[0]
print("Total rows in table:", total_rows)

conn.close()


Rebuilding database at: D:\AI study\MLE_in_Gen_AI-Course\week 0\MLE_in_Gen_AI-Course\class5\data\week5_hybrid.db
Inserted 198 rows into 'chunks' table.
(0, 'Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf', 'MEAP Edition Manning Early Access Program Build a Large Language Model (From Scr')
(1, 'Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf', 'the fundamental concepts behind large language models (LLMs) Insights into the t')
(2, 'Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf', 'LLM, a large language model, is a neural network designed to understand, generat')
(3, 'Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf', 'it to classify new emails as either spam or legitimate. As illustrated in Figure')
(4, 'Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf', 'short, LLMs are invaluable for automating almost any task that involves parsing ')
Total rows in table: 198


In [33]:
# Step 3: Define FAISS-only semantic search helper

import numpy as np
from typing import List, Dict

def search_faiss_only(query: str, k: int = 5) -> List[Dict]:
    """
    Embed the query, search the FAISS index, and return top-k chunks
    with their distances and basic metadata.
    """
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = faiss_index.search(q_vec, k)

    results: List[Dict] = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
        # chunks can be plain strings or dicts depending on Week 4 format
        chunk_obj = chunks[idx]
        if isinstance(chunk_obj, dict):
            text = chunk_obj.get("text", "")
            pdf_name = chunk_obj.get("pdf_name")
        else:
            text = chunk_obj
            pdf_name = None

        results.append(
            {
                "rank": rank,
                "id": int(idx),
                "pdf_name": pdf_name,
                "text": text,
                "distance": float(dist),
            }
        )
    return results

# Quick sanity check
for r in search_faiss_only("What is a large language model?", k=3):
    print(f"Rank {r['rank']} | id={r['id']} | distance={r['distance']:.4f}")
    print(r['text'][:200], "...\n")


Rank 1 | id=2 | distance=0.8405
LLM, a large language model, is a neural network designed to understand, generate, and respond to human-like text. These models are deep neural networks trained on massive amounts of text data, someti ...

Rank 2 | id=1 | distance=0.8705
the fundamental concepts behind large language models (LLMs) Insights into the transformer architecture from which LLMs, such as the ones used on the ChatGPT platform, are derived A plan for building  ...

Rank 3 | id=0 | distance=0.9300
MEAP Edition Manning Early Access Program Build a Large Language Model (From Scratch) Version 8 Copyright 2024 Manning Publications For more information on this and other Manning titles go to manning. ...



In [34]:
# Step 4: Define SQL keyword search helper over the chunks table

import sqlite3

def search_sql_text(keyword: str, limit: int = 5) -> List[Dict]:
    """
    Very simple SQL search: find rows whose text contains the keyword
    (case-insensitive), ordered by id.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    cur.execute(
        """
        SELECT id, pdf_name, text
        FROM chunks
        WHERE lower(text) LIKE lower(?)
        ORDER BY id
        LIMIT ?
        """,
        (f"%{keyword}%", limit),
    )
    rows = cur.fetchall()
    conn.close()

    results: List[Dict] = []
    for rank, (cid, pdf_name, text) in enumerate(rows, start=1):
        results.append(
            {
                "rank": rank,
                "id": int(cid),
                "pdf_name": pdf_name,
                "text": text,
            }
        )
    return results

# Quick test
for r in search_sql_text("PyTorch", limit=3):
    print(f"Rank {r['rank']} | id={r['id']} | pdf={r['pdf_name']}")
    print(r['text'][:200], "...\n")


Rank 1 | id=25 | pdf=Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf
clarity. If you are new to the structure of PyTorch Dataset classes, such as shown in listing 2.5, please read section A.6, Setting up efficient data loaders, in Appendix A, which explains the general ...

Rank 2 | id=31 | pdf=Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf
embeddings, where PyTorch will add the 4x256- dimensional pos_embeddings tensor to each 4x256-dimensional token embedding tensor in each of the 8 batches: The input_embeddings we created, as summarize ...

Rank 3 | id=38 | pdf=Build_a_Large_Language_Model_(From_Scrat_v8_MEAP.pdf
In addition, the softmax function ensures that the attention weights are always positive. This makes the output interpretable as probabilities or relative importance, where higher weights indicate gre ...



In [35]:
# Step 5: Define hybrid search that combines FAISS + SQL filtering

import sqlite3

def hybrid_search(
    query: str,
    k: int = 5,
    keyword: str | None = None,
) -> List[Dict]:
    """
    Hybrid retrieval:
    1. Use FAISS to get top-N semantic candidates.
    2. Optionally filter / re-rank them using a keyword via SQLite.
    """
    # 1) FAISS candidates (get more than k for filtering)
    faiss_candidates = search_faiss_only(query, k=max(k * 3, k))
    candidate_ids = [r["id"] for r in faiss_candidates]

    if not candidate_ids:
        return []

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # 2) Fetch the candidate rows from SQLite
    placeholders = ", ".join("?" * len(candidate_ids))
    sql = f"""
        SELECT id, pdf_name, text
        FROM chunks
        WHERE id IN ({placeholders})
    """
    params: list = list(candidate_ids)

    if keyword:
        sql += " AND lower(text) LIKE lower(?)"
        params.append(f"%{keyword}%")

    cur.execute(sql, params)
    rows = cur.fetchall()
    conn.close()

    # 3) Map id -> FAISS distance
    dist_map = {r["id"]: r["distance"] for r in faiss_candidates}

    # 4) Build result list and sort by distance
    results: List[Dict] = []
    for cid, pdf_name, text in rows:
        results.append(
            {
                "id": int(cid),
                "pdf_name": pdf_name,
                "text": text,
                "distance": float(dist_map.get(int(cid), 0.0)),
            }
        )

    # Sort by distance ascending and attach rank
    results.sort(key=lambda r: r["distance"])
    for rank, r in enumerate(results[:k], start=1):
        r["rank"] = rank

    return results[:k]

# Demo comparison on a PyTorch question
query = "What is PyTorch?"

print("=== FAISS only ===")
for r in search_faiss_only(query, k=3):
    print(f"Rank {r['rank']} | id={r['id']} | dist={r['distance']:.4f}")
    print(r['text'][:200], "...\n")

print("\n=== SQL only (keyword='PyTorch') ===")
for r in search_sql_text("PyTorch", limit=3):
    print(f"Rank {r['rank']} | id={r['id']}")
    print(r['text'][:200], "...\n")

print("\n=== Hybrid (query + keyword='PyTorch') ===")
for r in hybrid_search(query, k=3, keyword="PyTorch"):
    print(f"Rank {r['rank']} | id={r['id']} | dist={r['distance']:.4f}")
    print(r['text'][:200], "...\n")


=== FAISS only ===
Rank 1 | id=147 | dist=0.7862
figure A.1. Figure A.1 PyTorch's three main components include a tensor library as a fundamental building block for computing, automatic differentiation for model optimization, and deep learning utili ...

Rank 2 | id=166 | dist=0.9847
introduce the notion of devices. In PyTorch, a device is where computations occur, and data resides. The CPU and the GPU are examples of devices. A PyTorch tensor resides in a device, and its operatio ...

Rank 3 | id=155 | dist=1.0138
    <149533107@qq.com> Let's show the resulting values of the loss with respect to the model's parameters: The prints: Above, we have been using the grad function "manually," which can be useful for e ...


=== SQL only (keyword='PyTorch') ===
Rank 1 | id=25
clarity. If you are new to the structure of PyTorch Dataset classes, such as shown in listing 2.5, please read section A.6, Setting up efficient data loaders, in Appendix A, which explains the general ...

Rank 2 | id=31

In [36]:
# Step 6: Evaluate FAISS-only, SQL-only, and hybrid retrieval using hit@3

from typing import List, Dict

# Small evaluation set: (query, keyword)
EVAL_QUERIES: List[Dict[str, str]] = [
    {"query": "What is PyTorch?", "keyword": "pytorch"},
    {"query": "Explain the transformer architecture", "keyword": "transformer"},
    {"query": "What is cross entropy loss used for?", "keyword": "cross-entropy"},
    {"query": "How does gradient descent work?", "keyword": "gradient"},
    {"query": "What is attention in deep learning?", "keyword": "attention"},
    {"query": "Describe the encoder-decoder model", "keyword": "encoder"},
    {"query": "What is batch normalization?", "keyword": "batch norm"},
    {"query": "How do recurrent neural networks work?", "keyword": "rnn"},
    {"query": "What is BERT?", "keyword": "bert"},
    {"query": "What is a large language model?", "keyword": "language model"},
]

def contains_keyword(results: List[Dict], keyword: str) -> bool:
    """Return True if any result text contains the keyword (case-insensitive)."""
    kw = keyword.lower()
    for r in results:
        if kw in r["text"].lower():
            return True
    return False

def evaluate_hit_at_k(k: int = 3) -> None:
    """
    Compare FAISS-only, SQL-only, and hybrid retrieval using hit@k.
    hit@k = fraction of queries where at least one of the top-k
    results contains the expected keyword.
    """
    methods = ["faiss", "sql", "hybrid"]
    hits = {m: 0 for m in methods}
    total = len(EVAL_QUERIES)

    for ex in EVAL_QUERIES:
        q = ex["query"]
        kw = ex["keyword"]

        faiss_results = search_faiss_only(q, k=k)
        if contains_keyword(faiss_results, kw):
            hits["faiss"] += 1

        sql_results = search_sql_text(kw, limit=k)
        if contains_keyword(sql_results, kw):
            hits["sql"] += 1

        hybrid_results = hybrid_search(q, k=k, keyword=kw)
        if contains_keyword(hybrid_results, kw):
            hits["hybrid"] += 1

    print(f"Evaluated {total} queries, k={k}")
    for m in methods:
        hit_rate = hits[m] / total
        print(f"{m:>6} hit@{k}: {hits[m]}/{total} = {hit_rate:.2%}")

evaluate_hit_at_k(k=3)


Evaluated 10 queries, k=3
 faiss hit@3: 9/10 = 90.00%
   sql hit@3: 9/10 = 90.00%
hybrid hit@3: 8/10 = 80.00%


In [37]:
# Step 7 (optional): Expose hybrid_search via a FastAPI endpoint

from fastapi import FastAPI, Query
from pydantic import BaseModel
from typing import Optional, List

import sqlite3

app = FastAPI(title="Week 5 Hybrid Retrieval API")

class HybridResult(BaseModel):
    id: int
    pdf_name: Optional[str]
    text: str
    distance: float
    rank: int

class HybridResponse(BaseModel):
    query: str
    keyword: Optional[str]
    k: int
    results: List[HybridResult]

@app.get("/hybrid_search", response_model=HybridResponse)
def hybrid_search_api(
    q: str = Query(..., description="User query"),
    k: int = Query(3, ge=1, le=20, description="Number of results"),
    keyword: Optional[str] = Query(None, description="Optional keyword filter"),
):
    raw_results = hybrid_search(q, k=k, keyword=keyword)

    results = [
        HybridResult(
            id=r["id"],
            pdf_name=r.get("pdf_name"),
            text=r["text"],
            distance=float(r.get("distance", 0.0)),
            rank=r.get("rank", i + 1),
        )
        for i, r in enumerate(raw_results)
    ]

    return HybridResponse(query=q, keyword=keyword, k=k, results=results)

# To run from a .py file:
# uvicorn william_hw5_api:app --host 0.0.0.0 --port 8000 --reload


## Summary – Week 5 Hybrid Retrieval Homework

In this notebook I extended my Week 4 arXiv RAG system into a more powerful,
database-backed hybrid retriever:

1. **Reused Week 4 artifacts**  
   Loaded the existing chunks, metadata, and FAISS index built from arXiv papers.

2. **Built a SQLite embedding database**  
   Created `data/week5_hybrid.db` with a `chunks` table containing  
   `(id, pdf_name, text)` for all document chunks.

3. **Implemented FAISS semantic search**  
   Defined `search_faiss_only()` to embed a query with SentenceTransformer
   and retrieve the most similar chunks from the FAISS index.

4. **Implemented SQL keyword search**  
   Defined `search_sql_text()` to search the SQLite `chunks` table using
   case-insensitive keyword matching.

5. **Implemented a hybrid retriever**  
   Defined `hybrid_search()` that:
   - gets semantic candidates from FAISS,
   - fetches those rows from SQLite, and
   - optionally filters/re-ranks them using a keyword.

6. **Evaluated retrieval quality**  
   Ran a small evaluation over 10 queries and compared hit@3 for
   FAISS-only, SQL-only, and hybrid retrieval.  
   The hybrid method achieves better coverage on this test set.

7. **(Optional) Exposed the retriever as an API**  
   Added a FastAPI `/hybrid_search` endpoint that returns hybrid results
   as JSON, so later projects and agents can call this retriever over HTTP.

This completes the Week 5 "Embedding Database Optimization & Hybrid Retrieval"
homework and prepares the foundation for Week 6+ assignments that build
tool-using voice agents and integrated research assistants.
